In [1]:
import sys
import numpy as np
import cv2
import dlib
import face_recognition as fr

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("dlib:", dlib.__version__)
print("face_recognition:", fr.__version__)

Python: 3.10.20 | packaged by conda-forge | (main, Jun 11 2026, 03:28:51) [MSC v.1944 64 bit (AMD64)]
NumPy: 1.26.4
OpenCV: 4.8.1
dlib: 19.24.2
face_recognition: 1.2.3


In [2]:
import os
import csv
from datetime import datetime

In [3]:
import cv2
import face_recognition as fr
import numpy as np

In [6]:
known_face_encodings = []
known_face_names = []

people = [
    ("Chaitra", r"E:\All My Projects\Projects\Face Recognition-Based Attendance System\Images\Chaitra.jpg"),
    ("Kanchana", r"E:\All My Projects\Projects\Face Recognition-Based Attendance System\Images\Kanchana.jpeg"),
    ("Srinivas", r"E:\All My Projects\Projects\Face Recognition-Based Attendance System\Images\Srinivas.jpeg"),
    ("Vamsi", r"E:\All My Projects\Projects\Face Recognition-Based Attendance System\Images\Vamsi.jpeg")
]

for name, image_path in people:
    image = fr.load_image_file(image_path)
    encodings = fr.face_encodings(image)

    if len(encodings) > 0:
        known_face_encodings.append(encodings[0])
        known_face_names.append(name)

In [7]:
def mark_attendance(name):

    file_name = "attendance.csv"

    today = datetime.now().strftime("%d-%m-%Y")
    current_time = datetime.now().strftime("%H:%M:%S")

    # Create CSV if it doesn't exist
    if not os.path.exists(file_name):

        with open(file_name, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Name", "Date", "Time"])

    already_marked = False

    with open(file_name, "r", newline="") as f:

        reader = csv.reader(f)

        next(reader)

        for row in reader:

            if len(row) >= 2:

                if row[0] == name and row[1] == today:
                    already_marked = True
                    break

    if not already_marked:

        with open(file_name, "a", newline="") as f:

            writer = csv.writer(f)

            writer.writerow([name, today, current_time])

        print(f"Attendance Marked : {name}")

In [ ]:
# Webcam
# ==============================

video_capture = cv2.VideoCapture(0)

while True:

    ret, frame = video_capture.read()

    if not ret:
        break

    # Convert to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect Faces
    face_locations = fr.face_locations(rgb_frame)

    # Encode Faces
    face_encodings = fr.face_encodings(rgb_frame, face_locations)

    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):

        name = "Unknown"

        matches = fr.compare_faces(known_face_encodings, face_encoding)

        face_distances = fr.face_distance(known_face_encodings, face_encoding)

        best_match_index = np.argmin(face_distances)

        if matches[best_match_index]:

            name = known_face_names[best_match_index]

            # Mark Attendance
            mark_attendance(name)

        # Draw Face Box
        cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 0), 2)

        # Draw Name Box
        cv2.rectangle(frame,
                      (left, bottom - 35),
                      (right, bottom),
                      (0, 255, 0),
                      cv2.FILLED)

        # Display Name
        cv2.putText(frame,
                    name,
                    (left + 6, bottom - 6),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    2)

    cv2.imshow("Face Attendance System", frame)

    # Press q to Quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()
cv2.destroyAllWindows()

Attendance Marked : Chaitra
Attendance Marked : Vamsi
